In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import shutil
from google.colab import files

# ==========================================
# 3. MODEL VE LORA KONFİGÜRASYONU
# ==========================================
max_seq_length = 2048
dtype = torch.bfloat16
load_in_4bit = False

print("[*] Llama-3 (8B) ağırlıkları yükleniyor...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# ==========================================
# 4. VERİ SETİ VE EĞİTİM (SFT)
# ==========================================
print("[*] Eğitim verisi okunuyor...")
dataset = load_dataset("json", data_files="FORMATTED_CHESS_TRAIN_DATA.jsonl", split="train")

print("[*] SFTTrainer başlatılıyor, eğitim (orta oyun) başlıyor...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4, # Yüksek VRAM
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = False,
        bf16 = True,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "llama3_baseline_outputs",
        push_to_hub = False,
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

# ==========================================
# 5. AĞIRLIKLARI KAYDETME VE YEDEKLEME
# ==========================================
print("[+] Eğitim tamamlandı! Şah mat. Ağırlıklar kaydediliyor...")
model_name = "llama3_chess_commentator_lora"
model.save_pretrained(model_name)
tokenizer.save_pretrained(model_name)


shutil.make_archive(model_name, 'zip', model_name)

print(f"[+] {model_name}.zip indiriliyor...")
files.download(f"{model_name}.zip")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
[*] Llama-3 (8B) ağırlıkları yükleniyor...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

...

[+] Eğitim tamamlandı! Şah mat. Ağırlıklar kaydediliyor...


Unsloth: Restored added_tokens_decoder metadata in llama3_chess_commentator_lora/tokenizer_config.json.


[+] llama3_chess_commentator_lora.zip indiriliyor...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from unsloth import FastLanguageModel
import torch
from google.colab import drive
import os

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/llama3_chess_commentator_lora.zip"
extract_path = "/content/chess_model"

if not os.path.exists(extract_path):
    print("[*] Model paketinden çıkarılıyor...")
    !unzip -q "{zip_path}" -d {extract_path}
    print("[+] Çıkarma işlemi tamamlandı.")

max_seq_length = 2048
dtype = torch.bfloat16
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = extract_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)

prompt_style = """Aşağıda, yerine getirmen gereken bir görev ve bu göreve bağlam sağlayan bir girdi bulunmaktadır. Görevi başarıyla tamamlayan profesyonel bir satranç spikeri yorumu yaz.

### Görev (Instruction):
Bir satranç spikeri gibi hamleyi teknik ve heyecanlı bir dille yorumla.

### Bağlam (Input):
{}

### Yorum (Output):
"""

test_input = "Hamle No: 21 | Sıra: Siyah | Önceki Hamle: Rd1 | Yapılan Hamle: Qe8 (Vezir) | Aşama: Orta Oyun | Kalite: İyi | Skor: -3.44 | Taktikler: Açmaz Oluşturma | Tehditler: queen | Yeme: Hayır | Durum: Normal"

inputs = tokenizer(
[
    prompt_style.format(test_input)
], return_tensors = "pt").to("cuda")

print("[*] Spiker analizi yapıyor...")
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
response = tokenizer.batch_decode(outputs)

print("\n" + "="*30)
print("SPİKER YORUMU:")
print(response[0].split("### Yorum (Output):")[1].strip())
print("="*30)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
[*] Model paketinden çıkarılıyor...
[+] Çıkarma işlemi tamamlandı.
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]